# Пайплайн обучения моделей прогноза

Обучение моделей кодом репозитория на синтетической полке. Клиентских данных в репозитории нет.

| Стадия | Модуль | Результат |
|---|---|---|
| 0 | — | Синтетическая полка вместо `shelf_rolling.parquet` |
| A | `prognoz.predict.ml_core` | HGB q0.70 и `stamp.json` |
| B | `prognoz.predict.catboost_core` | CatBoost q0.70 и `stamp.json` |
| C | Генератор OOF в ноутбуке | OOF Parquet |
| D | `scripts/train-stacked-guard.py` | Бандл: HGB, q67, q70, мета и манифест |
| E | `prognoz.predict.stacked_guard_core` | Правило `HGB+1`, загрузка по пину |

Вне ноутбука: ETL из 1С, календарь, округление, `day_guard` и публикация релиза в `predict/daily.py`.


## 0. Окружение

`PROGNOZ_ML_MODEL_DIR` — песочница: без неё модель пишется в боевую папку.`PROGNOZ_1C_WRITE=0` требует `train-stacked-guard.py`.

In [ ]:
import os, sys, json, hashlib, shutil
from datetime import date, timedelta
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PROJECT / "src" / "prognoz").is_dir(), f"Запускать из репозитория, а не из {PROJECT}"

SANDBOX = PROJECT / "notebooks" / "_sandbox"
shutil.rmtree(SANDBOX, ignore_errors=True)
(SANDBOX / "models").mkdir(parents=True)
(SANDBOX / "output").mkdir(parents=True)

# Песочница ДО импорта prognoz: пути читаются на импорте модуля.
os.environ["PROGNOZ_1C_WRITE"] = "0"
os.environ["PROGNOZ_ML_MODEL_DIR"] = str(SANDBOX / "models")
os.environ["APP_OUTPUT_ROOT"] = str(SANDBOX / "output")

sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "analysis" / "etl"))

import numpy as np
import pandas as pd
import sklearn, catboost, joblib

print("python  ", sys.version.split()[0])
for mod in (np, pd, sklearn, catboost, joblib):
    print(f"{mod.__name__:8}", mod.__version__)
print("\nпесочница:", SANDBOX)

Прод пинует Python 3.12.10 / sklearn 1.9.0 / catboost 1.2.10(`web/requirements.txt`). Расхождение версий выше означает, что обученное здесьдля прода не годится: `_needs_refit` забракует его по версии sklearn (см. A.4).

## 0.1 Синтетическая полка

Схема — контракт `catboost_core._safe_shelf`.Заложено: недельная сезонность (для `wd_*`), `lost_units > 0` (таргет —восстановленный спрос), `supply_gap`, незрелый хвост `wo_settled=False`,`data_missing`, новинки последних дней.

In [ ]:
POINTS = [f"point-{i}" for i in range(1, 6)]            # 5 синтетических точек
GRPS = ["Шаурма / Классическая", "Шаурма / Острая", "Напитки"]
SKUS = [(f"sku-{i:03d}", GRPS[i % len(GRPS)]) for i in range(30)]

START = date(2024, 11, 1)        # cutoffs.training_start_requested из манифеста
EXPERT_CUTOFF = date(2026, 8, 20)  # cutoffs.expert_train_cutoff из манифеста
SHELF_END = EXPERT_CUTOFF + timedelta(days=3)   # полка живёт до issue-дня
WO_WATERMARK = SHELF_END - timedelta(days=10)   # позже — акты списаний не пришли

rng = np.random.default_rng(7)
days = pd.date_range(START, SHELF_END, freq="D")

rows = []
for point_no, point in enumerate(POINTS):
    point_scale = 0.7 + 0.15 * point_no
    for sku_no, (sku, grp) in enumerate(SKUS):
        base = 3.0 + 0.7 * (sku_no % 7)
        price = 150.0 + 25.0 * (sku_no % 9)
        # каждая 10-я пара — новинка: появляется за 5 дней до конца полки
        first = days[-5] if sku_no % 10 == 9 else days[0]
        weekday_profile = 0.75 + 0.1 * rng.random(7)
        for day in days[days >= first]:
            season = weekday_profile[day.weekday()]
            trend = 1.0 + 0.15 * ((day - days[0]).days / max(len(days) - 1, 1))
            mu = base * point_scale * season * trend
            demand = max(0.0, rng.poisson(mu) + rng.normal(0, 0.4))
            # привоз ограничен: часть спроса не увидена продажами
            supply = np.floor(mu * (1.0 + 0.25 * rng.random()))
            sold = min(demand, supply)
            lost = max(0.0, demand - sold)
            gap = bool(rng.random() < 0.01)          # срыв поставки
            missing = bool(rng.random() < 0.004)     # выгрузка не пришла
            rows.append({
                "point_key": point, "sku_key": sku, "trade_day": day, "grp": grp,
                "sold_units": float(np.round(sold, 1)),
                "lost_units": float(np.round(lost, 1)),
                "u40_units": float(np.round(max(0.0, supply - demand) * 0.3, 1)),
                "wo_units": float(np.round(max(0.0, supply - demand) * 0.2, 1)),
                "avg_price": price,
                "supply_gap": gap,
                "wo_settled": bool(day.date() <= WO_WATERMARK),
                "data_missing": missing,
            })

shelf = pd.DataFrame(rows)
SHELF_PATH = SANDBOX / "shelf_rolling.parquet"
shelf.to_parquet(SHELF_PATH, index=False)

print(f"строк полки : {len(shelf):,}")
print(f"период      : {shelf['trade_day'].min().date()} … {shelf['trade_day'].max().date()}")
print(f"пар         : {shelf.groupby(['point_key','sku_key']).ngroups}")
print(f"supply_gap  : {int(shelf['supply_gap'].sum())}")
print(f"незрелых дней (wo_settled=False): {int((~shelf['wo_settled']).sum()):,}")
print(f"sha256      : {hashlib.sha256(SHELF_PATH.read_bytes()).hexdigest()[:16]}…")

`filter_frame(..., "metric_scope")` — чёрный список пар. Синтетических ключей вполитике нет, но проверяем: иначе панель окажется пустой.

In [ ]:
from prognoz.assortment import filter_frame, provenance as assortment_provenance

kept = filter_frame(shelf, "metric_scope", date_col="trade_day")
print(f"до фильтра: {len(shelf):,}  после: {len(kept):,}")
print("политика  :", assortment_provenance())
assert len(kept) == len(shelf), "политика вычла синтетические пары — проверьте ключи"

## A. Движок `ml` — HistGradientBoosting q0.70

`build_panel → add_features → snapshot_categories → apply_categories → fit →make_stamp → save_model`.`Q = 0.70`: недовоз стоит 0.67 маржи, перевоз 0.33 себестоимости. `LAG = 3` —горизонт D+2.

In [ ]:
from prognoz.predict import ml_core

TARGET = EXPERT_CUTOFF + timedelta(days=3)      # цель D+2 относительно issue-дня
cfg_ml = ml_core.MlCfg()                        # lag=3, quantile=0.70, refit_days=7
CUTOFF = TARGET - timedelta(days=cfg_ml.lag)    # обучение не позже target − lag

print("FEATS_VERSION :", ml_core.FEATS_VERSION)
print("Q / LAG       :", ml_core.Q, "/", ml_core.LAG)
print("GBM_PARAMS    :", ml_core.GBM_PARAMS)
print("DROP_FEATS    :", ml_core.DROP_FEATS)
print("target / cutoff:", TARGET, "/", CUTOFF)

### A.1 Панель и признаки

Все окна считаются по сдвинутой на `lag` серии и кончаются на `D − lag`.`DROP_FEATS` убирает `month`, `week` (нулевая важность) и `wo_rate28`,`surplus_l` (на свежем хвосте занижены непришедшими актами списаний).

In [ ]:
excluded = ml_core.load_excluded()
print(f"нетипичных дней в analysis/config/excluded_days.json: {len(excluded)}")

pan_ml = ml_core.build_panel(shelf, TARGET, excluded=excluded)
pan_ml, feats_ml = ml_core.add_features(pan_ml, cfg_ml.lag)

print(f"\nстрок панели : {len(pan_ml):,}  (включая будущие дни с dem=NaN)")
print(f"строк с меткой: {int(pan_ml['dem'].notna().sum()):,}")
print(f"признаков    : {len(feats_ml)}")
for name in feats_ml:
    print("   ", name)

### A.2 Причинность

Портим хвост полки после `cutoff` — признаки целевого дня не должны измениться.Плюс положительный контроль: порча самого дня `cutoff` изменить их обязана.

In [ ]:
probe_cols = ["lag3", "r28_mean", "wd_mean", "price_r28"]
before = (pan_ml[pan_ml["day"] == pd.Timestamp(TARGET)]
          .set_index(["point_key", "sku_key"])[probe_cols].sort_index())

def features_after_taint(mask):
    tainted = shelf.copy()
    tainted.loc[mask, ["sold_units", "lost_units"]] *= 100.0
    pan_t, _ = ml_core.add_features(
        ml_core.build_panel(tainted, TARGET, excluded=excluded), cfg_ml.lag)
    return (pan_t[pan_t["day"] == pd.Timestamp(TARGET)]
            .set_index(["point_key", "sku_key"])[probe_cols].sort_index())

after = features_after_taint(shelf["trade_day"] > pd.Timestamp(CUTOFF))
same = before.equals(after)
print("порча ПОСЛЕ cutoff  → признаки идентичны:", same)
assert same, "УТЕЧКА: признак увидел данные позже cutoff"

# Положительный контроль: тест, который никогда не срабатывает, ничего не
# доказывает. Портим САМ день cutoff — признаки обязаны измениться.
control = features_after_taint(shelf["trade_day"] == pd.Timestamp(CUTOFF))
changed = [c for c in probe_cols if not before[c].equals(control[c])]
print("порча дня cutoff    → изменились:", changed or "ничего")
assert changed, "тест вырожден: он не заметил бы и настоящую утечку"

### A.3 Категории, обучение, штамп

Словарь категорий уезжает в штамп: незнакомая точка на инференсе → `NaN`(missing для HistGB), а не чужой код.

In [ ]:
cats = ml_core.snapshot_categories(pan_ml)
pan_ml = ml_core.apply_categories(pan_ml, cats)
print("точек в словаре:", len(cats["points"]), " групп:", len(cats["grps"]))

model_ml, train_ml = ml_core.fit(pan_ml, feats_ml, CUTOFF, cfg_ml)
stamp_ml = ml_core.make_stamp(feats_ml, CUTOFF, cfg_ml, train_ml, cats)
ml_core.save_model(model_ml, stamp_ml, cfg_ml)

print(f"\nобучено на {len(train_ml):,} строк, последняя метка {train_ml['day'].max().date()}")
print("артефакты:", sorted(p.name for p in ml_core.model_dir(cfg_ml.lag).iterdir()))
print("\nштамп:")
print(json.dumps({k: v for k, v in stamp_ml.items() if k != "categories"},
                 ensure_ascii=False, indent=2))

### A.4 Правила переобучения

Отдельная ветка — отрицательный возраст: модель, обученная позже цели. Без неёбэктест выглядел бы безупречно, потому что подсматривал ответ.

In [ ]:
checks = {
    "свежая модель, возраст 0 дн": ml_core._needs_refit(stamp_ml, CUTOFF, cfg_ml),
    "возраст 7 дн (порог)": ml_core._needs_refit(
        stamp_ml, CUTOFF + timedelta(days=7), cfg_ml),
    "модель из будущего": ml_core._needs_refit(
        stamp_ml, CUTOFF - timedelta(days=1), cfg_ml),
    "сменилась версия фич": ml_core._needs_refit(
        {**stamp_ml, "feats_version": "ml-old"}, CUTOFF, cfg_ml),
    "сменился sklearn": ml_core._needs_refit(
        {**stamp_ml, "sklearn_version": "0.0.0"}, CUTOFF, cfg_ml),
    "сменилась политика ассортимента": ml_core._needs_refit(
        {**stamp_ml, "assortment_policy_sha256": "x" * 64}, CUTOFF, cfg_ml),
}
for label, reason in checks.items():
    print(f"{label:35} → {reason or 'модель годна'}")

### A.5 Предсказание и пол новинок

`novelty_floor` ставит пол, а не заменяет расчёт: уровень SKU на других точках,масштаб точки, гашение по мере накопления истории. «8 штук всем новинкам» в лобна бэктесте ухудшало на 175–299 тыс ₽/июль.

In [ ]:
# pan_ml построена по полке до TARGET и годится для обучения (метки маскированы),
# но n_obs_days по ней посчитался бы ВКЛЮЧАЯ дни после cutoff — а он кормит вес
# гашения пола новинок. Прод видит полку, обрезанную на cutoff: считаем по ней.
pan_issue, _ = ml_core.add_features(
    ml_core.build_panel(shelf[shelf["trade_day"] <= pd.Timestamp(CUTOFF)],
                        TARGET, excluded=excluded), cfg_ml.lag)
pan_issue = ml_core.apply_categories(pan_issue, stamp_ml["categories"])

preds_ml = ml_core.predict_rows(model_ml, stamp_ml, pan_issue, TARGET)
preds_ml = ml_core.novelty_floor(preds_ml, pan_issue, TARGET, cfg_ml.lag)

print(f"строк прогноза: {len(preds_ml)}   новинок: {int(preds_ml['is_novelty'].sum())}")
print(f"mu_ml: медиана {preds_ml['mu_ml'].median():.1f}, сумма {preds_ml['mu_ml'].sum():.0f}")
display(preds_ml.sort_values("is_novelty", ascending=False).head(8))
print("\nпояснение строки:", ml_core.explain_row(preds_ml.iloc[0]))

## B. Движок `catboost` — CatBoost Quantile α=0.70

Другая панель. Двойная маска:| | маскируется | становится ||---|---|---|| `dem` (метка) | `supply_gap ∨ data_missing ∨ ¬wo_settled ∨ excluded` | `NaN` || `feature_dem` (признак) | `supply_gap ∨ data_missing ∨ excluded` | на незрелом хвосте — `sold_units` |

In [ ]:
from prognoz.predict import catboost_core

cfg_cat = catboost_core.CatCfg(lag=3)      # quantile=0.70, iterations=700, depth=7
print("FEATURES_VERSION:", catboost_core.FEATURES_VERSION)
print("признаков       :", len(catboost_core.NUMERIC_FEATURES), "+",
      len(catboost_core.CAT_FEATURES), "категориальных")
print("конфиг          :", cfg_cat)

model_cat, stamp_cat = catboost_core.train(
    shelf, CUTOFF, cfg=cfg_cat, excluded=excluded, save=True)

print(f"\nобучено на {stamp_cat['n_train_rows']:,} строк по {stamp_cat['train_cutoff']}")
print("артефакты:", sorted(p.name for p in catboost_core.model_dir(cfg_cat.lag).iterdir()))
print("\nштамп:")
print(json.dumps(stamp_cat, ensure_ascii=False, indent=2))

### B.1 Двойная маска на данных


In [ ]:
pan_cat = catboost_core.build_panel(
    shelf, EXPERT_CUTOFF, as_of=EXPERT_CUTOFF, excluded=excluded, target_mode="recovered")

tail = pan_cat[pan_cat["day"] > pd.Timestamp(WO_WATERMARK)]
print(f"дней после водяного знака {WO_WATERMARK}: строк {len(tail):,}")
print(f"  метка dem задана        : {int(tail['dem'].notna().sum()):,}  (ожидаем 0)")
print(f"  признак feature_dem есть: {int(tail['feature_dem'].notna().sum()):,}  (ожидаем много)")

mature = pan_cat[pan_cat["day"] <= pd.Timestamp(WO_WATERMARK)]
print(f"\nна зрелых днях метка задана: {int(mature['dem'].notna().sum()):,} из {len(mature):,}")
assert tail["dem"].notna().sum() == 0

## C. Восстановленное звено: генератор OOF

`train-stacked-guard.py --oof` ждёт parquet, которого в репозитории нет:провенанс-колонки (`hgb_source_sha256`, `q67_block_id`, `*_feature_as_of`)встречаются только в самом скрипте и его тесте. В проде это внешний файл`PROGNOZ_STACKED_GUARD_OOF`.Схема отвечает на вопрос «что видел бы ночной прогон в тот день». Тритребования:1. Признаки — из снимка полки на день выпуска (включая зрелость списаний,   известную тогда). Метка — из финальной полки.2. Мета-признаки — тем же кодом, что `predict_bundle`: округление **до**   разностей, `expert_mean` по двум CatBoost без HGB.3. Эксперты блоков — прод-конфигурации, иначе остатки другой формы.Прогон: переобучение раз в `REFIT_DAYS = 7` дней, признаки из снимка каждого дня.

In [ ]:
import importlib.util
from sklearn.ensemble import HistGradientBoostingRegressor
from prognoz.predict import stacked_guard_core as core

spec = importlib.util.spec_from_file_location(
    "train_stacked_guard", PROJECT / "scripts" / "train-stacked-guard.py")
tsg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tsg)

LAG = core.BUNDLE_LAG                  # 3 — горизонт D+2
REFIT_DAYS = ml_core.REFIT_DAYS        # 7 — шаг переобучения, как в проде
OOF_START = date(2026, 5, 1)
OOF_END = date(2026, 7, 30)            # cutoffs.meta_label_cutoff из манифеста

# Лаг зрелости списаний: в генераторе полка кончается 23.08, водяной знак 13.08.
# На реальной полке это расстояние между её максимумом и max(trade_day) с
# wo_settled=True; в прод-манифесте оно равно одному дню (08-19 против 08-20).
WO_SETTLE_LAG = (SHELF_END - WO_WATERMARK).days

SHELF_SHA256 = hashlib.sha256(SHELF_PATH.read_bytes()).hexdigest()
print("источник OOF (hgb_source_sha256):", SHELF_SHA256[:16], "…")
print(f"окно OOF      : {OOF_START} … {OOF_END} ({(OOF_END - OOF_START).days + 1} дней)")
print(f"переобучение  : каждые {REFIT_DAYS} дн, итерации CatBoost 500/508 (прод)")
print(f"лаг зрелости  : {WO_SETTLE_LAG} дн")

### C.1 Снимок полки на день выпуска

Ночной прогон дня `D` видит полку до `D − 3`, и списания в ней закрыты сотставанием. Одна панель на конец истории подарила бы марту знание августа.Замер ниже.

In [ ]:
def shelf_snapshot(source: pd.DataFrame, as_of: date) -> pd.DataFrame:
    """Полка, какой её видел ночной прогон с информационной границей as_of."""
    snap = source[source["trade_day"] <= pd.Timestamp(as_of)].copy()
    snap["wo_settled"] = (
        snap["trade_day"] <= pd.Timestamp(as_of - timedelta(days=WO_SETTLE_LAG)))
    return snap


probe_day = date(2026, 6, 1)
probe_as_of = probe_day - timedelta(days=LAG)

naive = tsg.full_panel(shelf, probe_day)                        # одна панель на всё
honest = tsg.full_panel(shelf_snapshot(shelf, probe_as_of), probe_day)

def at(panel, day):
    return (panel[panel["day"] == pd.Timestamp(day)]
            .set_index(["point_key", "sku_key"])["lag3"].sort_index())

a, b = at(naive, probe_day), at(honest, probe_day)
gap = (a - b).dropna()
print(f"lag3 на {probe_day}: панель на конец истории против снимка на {probe_as_of}")
print(f"  строк сопоставлено      : {len(gap)}")
print(f"  завышение в среднем     : {gap.mean():+.2f} шт")
print(f"  строк с завышением > 0  : {int((gap > 0).sum())} из {len(gap)}")
print(f"  относительно величины   : {100 * gap.sum() / a.reindex(gap.index).sum():.1f}%")

`feature_dem` на «созревших» днях включает потерянный спрос, не измеримый намомент выпуска. Дальше — только снимки.

### C.2 Мета-признаки — один код с инференсом

Порядок как в `predict_bundle`: `_half_up_nonnegative`, затем разности, затем`expert_mean = 0.5 × (q70 + q67)`. Сверка с инференсом — в C.7.

In [ ]:
def meta_features(frame: pd.DataFrame, hgb_raw, q67_raw, q70_raw) -> pd.DataFrame:
    """Те же семь производных признаков, что строит predict_bundle.

    Порядок критичен: округление ДО разностей. Иначе мета учится на
    непрерывной величине, а на инференсе получает целые.
    """
    hgb = core._half_up_nonnegative(hgb_raw)
    q67 = core._half_up_nonnegative(q67_raw)
    q70 = core._half_up_nonnegative(q70_raw)
    out = frame.copy()
    out["hgb_qty"], out["q67_qty"], out["q70_qty"] = hgb, q67, q70
    out["q70_minus_hgb"] = q70 - hgb
    out["q67_minus_hgb"] = q67 - hgb
    out["q70_minus_q67"] = q70 - q67
    out["expert_mean"] = 0.5 * (q70 + q67)       # без HGB — как в predict_bundle
    return out


def fit_experts(train_frame: pd.DataFrame, categories: dict):
    """Три эксперта прод-конфигурации: HGB q0.70, CatBoost q0.67/500, q0.70/508."""
    hgb = HistGradientBoostingRegressor(
        loss="quantile", quantile=0.70,
        categorical_features=["point_cat", "grp_cat"], **ml_core.GBM_PARAMS)
    hgb.fit(tsg.hgb_frame(train_frame, categories), train_frame["dem"])
    expert_x = tsg.cat_frame(train_frame, core.EXPERT_NUMERIC, core.EXPERT_CATEGORICAL)
    q67 = tsg.cat_model(0.67, 500)
    q70 = tsg.cat_model(0.70, 508)
    q67.fit(expert_x, train_frame["dem"], cat_features=core.EXPERT_CATEGORICAL)
    q70.fit(expert_x, train_frame["dem"], cat_features=core.EXPERT_CATEGORICAL)
    return hgb, q67, q70


print("meta_features и fit_experts определены")

### C.3 Метки — из зрелых данных

Метка берётся из финальной полки: к моменту сборки бандла спрос за прошлые дниизвестен, списания закрыты. Это и есть смысл `meta_cutoff`.

In [ ]:
labels = tsg.full_panel(shelf, SHELF_END)
labels = labels[labels["dem"].notna()][
    ["point_key", "sku_key", "day", "dem", "avg_price"]].rename(
    columns={"dem": "demand", "avg_price": "price"})
labels = labels[(labels["day"] >= pd.Timestamp(OOF_START))
                & (labels["day"] <= pd.Timestamp(OOF_END))]

print(f"зрелых меток в окне OOF: {len(labels):,}")
print(f"последний зрелый день  : {labels['day'].max().date()} "
      f"(водяной знак финальной полки {WO_WATERMARK})")

### C.4 Прогон по дням выпуска

На каждый день `D`: `as_of = D − 3`, снимок, панель; раз в `REFIT_DAYS` —переобучение трёх экспертов на `day ≤ as_of`; предсказание строк `D`;провенанс `train_cutoff` = `as_of` последнего переобучения, `feature_as_of` =`D − 3`.Модель старше признаков — как в проде, где на диске модель последнегопереобучения, а полка свежая.

In [ ]:
import time

issue_days = pd.date_range(OOF_START, OOF_END, freq="D").date
started = time.time()
experts = None
refit_as_of = None
refit_categories = None
refit_no = 0
oof_parts = []

for day in issue_days:
    as_of = day - timedelta(days=LAG)
    snap = shelf_snapshot(shelf, as_of)
    panel = tsg.full_panel(snap, day)

    stale = refit_as_of is None or (as_of - refit_as_of).days >= REFIT_DAYS
    if stale:
        train_frame = panel[
            panel["dem"].notna()
            & (panel["day"] >= pd.Timestamp(START))
            & (panel["day"] <= pd.Timestamp(as_of))
        ].copy()
        refit_categories = {
            "points": sorted(train_frame["point_key"].dropna().astype(str).unique().tolist()),
            "grps": sorted(train_frame["grp"].dropna().astype(str).unique().tolist()),
        }
        experts = fit_experts(train_frame, refit_categories)
        refit_as_of = as_of
        refit_no += 1
        print(f"  переобучение {refit_no:2} на {as_of}: строк {len(train_frame):,}")

    hgb, q67, q70 = experts
    rows = panel[panel["day"] == pd.Timestamp(day)].copy()
    if rows.empty:
        continue
    expert_x = tsg.cat_frame(rows, core.EXPERT_NUMERIC, core.EXPERT_CATEGORICAL)
    rows = meta_features(
        rows,
        hgb.predict(tsg.hgb_frame(rows, refit_categories)),
        q67.predict(expert_x),
        q70.predict(expert_x),
    )
    block_id = f"refit-{refit_no:02d}-{refit_as_of.isoformat()}"
    rows = rows.assign(
        hgb_train_cutoff=pd.Timestamp(refit_as_of),
        q67_train_cutoff=pd.Timestamp(refit_as_of),
        q70_train_cutoff=pd.Timestamp(refit_as_of),
        hgb_feature_as_of=pd.Timestamp(as_of),
        q67_feature_as_of=pd.Timestamp(as_of),
        q70_feature_as_of=pd.Timestamp(as_of),
        q67_block_id=block_id, q70_block_id=block_id,
        hgb_source_sha256=SHELF_SHA256,
    )
    oof_parts.append(rows.drop(columns=["dem"], errors="ignore"))

raw_oof = pd.concat(oof_parts, ignore_index=True)
print(f"\nдней выпуска {len(issue_days)}, переобучений {refit_no}, "
      f"строк до склейки с метками {len(raw_oof):,}, "
      f"время {(time.time() - started) / 60:.1f} мин")

### C.5 Склейка с метками и запись файла


In [ ]:
oof = raw_oof.merge(labels, on=["point_key", "sku_key", "day"], how="inner")
oof["price"] = pd.to_numeric(oof["price"], errors="coerce")
oof = oof[np.isfinite(oof["price"]) & (oof["price"] > 0)].copy()

required = sorted(set(
    core.META_NUMERIC + core.META_CATEGORICAL
    + ["point_key", "sku_key", "day", "demand", "price",
       "hgb_qty", "q67_qty", "q70_qty",
       "hgb_train_cutoff", "q67_train_cutoff", "q70_train_cutoff",
       "hgb_feature_as_of", "q67_feature_as_of", "q70_feature_as_of",
       "q67_block_id", "q70_block_id", "hgb_source_sha256"]))
missing = sorted(set(required) - set(oof.columns))
assert not missing, f"OOF не хватает колонок: {missing}"

oof = oof.drop_duplicates(["point_key", "sku_key", "day"]).sort_values(
    ["point_key", "sku_key", "day"]).reset_index(drop=True)

OOF_PATH = SANDBOX / "oof_with_hgb_provenance.parquet"
oof[required].to_parquet(OOF_PATH, index=False)

print(f"строк      : {len(oof):,}")
print(f"колонок    : {len(required)}")
print(f"период     : {oof['day'].min().date()} … {oof['day'].max().date()}")
print(f"блоков     : {oof['q67_block_id'].nunique()}")
print(f"\nостаток demand − hgb_qty: медиана {(oof['demand'] - oof['hgb_qty']).median():+.2f}")
print("q70_minus_hgb уникальных значений:", oof["q70_minus_hgb"].nunique(),
      "— целые, как на инференсе" if oof["q70_minus_hgb"].nunique() < 20 else "— НЕПРЕРЫВНЫЕ, дефект")

### C.6 Приёмка контракта

`load_oof` проверяет объявленный провенанс — это схема, не доказательствопричинности. Рядом отрицательный контроль: испорченный cutoff.

In [ ]:
frame, source_sha = tsg.load_oof(OOF_PATH, OOF_END)
print(f"load_oof принял файл: строк {len(frame):,}, "
      f"hgb_source_sha256 {source_sha[:16]}…")
print(f"  residual_label медиана: {frame['residual_label'].median():+.2f}")

bad = oof.copy()
bad["hgb_train_cutoff"] = bad["hgb_train_cutoff"] + pd.Timedelta(days=1)
BAD_PATH = SANDBOX / "oof_broken.parquet"
bad[required].to_parquet(BAD_PATH, index=False)
try:
    tsg.load_oof(BAD_PATH, OOF_END)
    print("\n!!! файл с моделью из будущего принят — это был бы дефект")
except RuntimeError as exc:
    print(f"\nиспорченный cutoff отклонён: {exc}")

### C.7 Сверка мета-признаков с `predict_bundle`

Расхождение здесь означало бы, что мета учится не на том, что получит в бою.

In [ ]:
check_day = date(2026, 7, 15)
check_as_of = check_day - timedelta(days=LAG)
snap = shelf_snapshot(shelf, check_as_of)
panel = tsg.full_panel(snap, check_day)
train_frame = panel[panel["dem"].notna()
                    & (panel["day"] <= pd.Timestamp(check_as_of))].copy()
cats_chk = {
    "points": sorted(train_frame["point_key"].dropna().astype(str).unique().tolist()),
    "grps": sorted(train_frame["grp"].dropna().astype(str).unique().tolist()),
}
hgb_c, q67_c, q70_c = fit_experts(train_frame, cats_chk)

rows = panel[panel["day"] == pd.Timestamp(check_day)].copy()
expert_x = tsg.cat_frame(rows, core.EXPERT_NUMERIC, core.EXPERT_CATEGORICAL)
mine = meta_features(rows,
                     hgb_c.predict(tsg.hgb_frame(rows, cats_chk)),
                     q67_c.predict(expert_x),
                     q70_c.predict(expert_x))

# Тот же расчёт руками кодом predict_bundle (stacked_guard_core.py:385-395)
h = core._half_up_nonnegative(hgb_c.predict(tsg.hgb_frame(rows, cats_chk)))
a = core._half_up_nonnegative(q67_c.predict(expert_x))
b = core._half_up_nonnegative(q70_c.predict(expert_x))
prod = pd.DataFrame({
    "hgb_qty": h, "q67_qty": a, "q70_qty": b,
    "q70_minus_hgb": b - h, "q67_minus_hgb": a - h, "q70_minus_q67": b - a,
    "expert_mean": 0.5 * (b + a),
}, index=mine.index)

same = [c for c in core.META_EXTRA_NUMERIC if mine[c].equals(prod[c])]
diff = [c for c in core.META_EXTRA_NUMERIC if not mine[c].equals(prod[c])]
print(f"совпало признаков: {len(same)} из {len(core.META_EXTRA_NUMERIC)}")
if diff:
    print("РАСХОЖДЕНИЕ:", diff)
assert not diff, "мета-признаки OOF не совпадают с инференсом"
print("мета-признаки OOF идентичны тем, что построит predict_bundle")

### C.8 Качество против наивных базовых линий

Деньги промаха `over × 0.33 + under × 0.67`, умноженные на цену. Базовые линииокруглены так же, как прогноз. Бутстрэп блочный: строки внутри одногопереобучения зависимы.

In [ ]:
demand_v = oof["demand"].to_numpy(float)
price_v = oof["price"].to_numpy(float)

def miss_money(pred):
    q = np.asarray(pred, dtype=float)
    over = np.clip(q - demand_v, 0, None)
    under = np.clip(demand_v - q, 0, None)
    return (over * 0.33 + under * 0.67) * price_v

rounded = core._half_up_nonnegative
variants = {
    "наив r28_mean": rounded(oof["r28_mean"].fillna(0)),
    "наив r28_q75": rounded(oof["r28_q75"].fillna(0)),
    "HGB q0.70": oof["hgb_qty"],
    "CatBoost q0.67": oof["q67_qty"],
    "CatBoost q0.70": oof["q70_qty"],
}
per_row = {name: miss_money(pred) for name, pred in variants.items()}
for name, pred in variants.items():
    bias = float(np.mean(np.asarray(pred, float) - demand_v))
    print(f"  {name:16} деньги промаха {per_row[name].sum() / 1000:9.1f} тыс   "
          f"смещение {bias:+.2f}")

gain = per_row["наив r28_q75"] - per_row["CatBoost q0.67"]
blocks_v = oof["q67_block_id"].to_numpy()
unique_blocks = np.unique(blocks_v)
rng_boot = np.random.default_rng(0)
boot = np.array([
    sum(gain[blocks_v == b].sum() for b in rng_boot.choice(unique_blocks, len(unique_blocks)))
    / 1000 for _ in range(2000)])
print(f"\nq67 против наива r28_q75: выигрыш {gain.sum() / 1000:.1f} тыс "
      f"({100 * gain.sum() / per_row['наив r28_q75'].sum():.2f}%)")
print(f"  блочный бутстрэп 95%: [{np.percentile(boot, 2.5):.1f}; "
      f"{np.percentile(boot, 97.5):.1f}] тыс по {len(unique_blocks)} блокам")
print(f"  строк, где модель лучше наива: {100 * (gain > 0).mean():.1f}% "
      "(выигрыш идёт из асимметрии 0.67/0.33, а не из частоты попаданий)")

## D. Бандл stacked_guard

`build_bundle`: панель и отбор `training_start … expert_cutoff` (мин. 1000строк) → словарь категорий → три эксперта (HGB q0.70 на `ml_core.GBM_PARAMS`,CatBoost q0.67/500, q0.70/508) → мета на OOF (таргет `demand − hgb_qty`,`Quantile:alpha=0.67`, 350 итераций, веса по цене) → временная папка, sha256,манифест, промоушен, повторная валидация.Предохранители: `PROGNOZ_1C_WRITE=0`, выход под `PROGNOZ_ML_MODEL_DIR`,существующая папка не перезаписывается.

In [ ]:
BUNDLE = Path(os.environ["PROGNOZ_ML_MODEL_DIR"]) / "stacked_guard" / "runs" / "notebook" / "lag_3"

manifest = tsg.build_bundle(
    shelf_path=SHELF_PATH,
    oof_path=OOF_PATH,
    output=BUNDLE,
    expert_cutoff=EXPERT_CUTOFF,
    meta_cutoff=OOF_END,
    training_start=START,
)
print("бандл собран:", BUNDLE)
print("файлы:", sorted(p.name for p in BUNDLE.iterdir()))

### D.1 Предохранители


In [ ]:
for label, kwargs in (
    ("папка уже существует", dict(output=BUNDLE)),
    ("выход вне PROGNOZ_ML_MODEL_DIR", dict(output=SANDBOX / "чужая" / "lag_3")),
):
    try:
        tsg.build_bundle(
            shelf_path=SHELF_PATH, oof_path=OOF_PATH,
            expert_cutoff=EXPERT_CUTOFF, meta_cutoff=OOF_END, training_start=START,
            **kwargs)
        print(f"{label:35} → !!! прошло, а не должно было")
    except (FileExistsError, RuntimeError) as exc:
        print(f"{label:35} → отклонено: {type(exc).__name__}: {exc}")

### D.2 Манифест

Схема признаков, словари категорий, cutoff-ы, гиперпараметры, sha256 входов,версии библиотек, sha256 каждого файла весов.

In [ ]:
def show(node, prefix=""):
    for key, value in node.items():
        if isinstance(value, dict) and key != "files":
            show(value, prefix + key + ".")
        else:
            text = str(value)
            print(f"{prefix + key:42} {text[:88]}{'…' if len(text) > 88 else ''}")

show({k: v for k, v in manifest.items() if k not in {"hgb_categories", "feature_schema"}})
print("\nfeature_schema.expert_numeric :", len(manifest["feature_schema"]["expert_numeric"]))
print("feature_schema.meta_numeric   :", len(manifest["feature_schema"]["meta_numeric"]))
print("hgb_categories.points         :", len(manifest["hgb_categories"]["points"]))
print("\nканонический sha256 манифеста:", core._manifest_digest(manifest))

## E. Правило консенсуса и загрузка бандла

```applied = (round(clip(meta, 0, 2)) >= 1) & (q67 >= hgb + 1) & (q70 >= hgb + 1)qty     = hgb + applied```Понизить HGB правило не может: при нарушении бросает `AssertionError`. Втораяединица провалила экономический тест.

In [ ]:
cases = pd.DataFrame([
    # hgb, q67, q70, meta_raw
    (7, 8, 8, 1.0),   # все согласны → +1
    (7, 8, 8, 0.4),   # мета против   → HGB
    (7, 7, 8, 1.0),   # q67 против    → HGB
    (7, 8, 7, 1.0),   # q70 против    → HGB
    (7, 9, 9, 1.9),   # мета просит 2 → всё равно +1
    (0, 1, 1, 1.0),   # ноль тоже поднимается на 1
], columns=["hgb", "q67", "q70", "meta_raw"])

qty, applied = core.consensus_add1(
    cases["hgb"], cases["q67"], cases["q70"], cases["meta_raw"])
cases["qty"] = qty
cases["applied"] = applied
display(cases)
assert (cases["qty"] >= cases["hgb"]).all(), "правило понизило HGB"

### E.1 Загрузка по пину, три горизонта

Непоставочный бандл без `PROGNOZ_STACKED_GUARD_MANIFEST_SHA256` не грузится,хеш проверяется до чтения весов. Один бандл `lag_3` обслуживает D+2/D+3/D+4:адаптер кладёт последний полный день продаж в слот `lag3`.

In [ ]:
os.environ["PROGNOZ_STACKED_GUARD_BUNDLE"] = str(BUNDLE)
os.environ["PROGNOZ_STACKED_GUARD_MANIFEST_SHA256"] = core._manifest_digest(manifest)

for horizon, lag in ((2, 3), (3, 4), (4, 5)):
    target = EXPERT_CUTOFF + timedelta(days=horizon + 1)
    bundle = core.load_bundle(target=target, cfg=core.GuardCfg(lag=lag))
    panel = core._full_feature_panel(shelf, target, lag)
    preds = core.predict_bundle(bundle, shelf, target, cfg=core.GuardCfg(lag=lag),
                                panel=panel)
    print(f"D+{horizon}  цель {target}  lag {lag}  "
          f"эффективный лаг {panel.attrs.get('effective_lag')}  "
          f"данные по {panel.attrs.get('data_as_of')}")
    print(f"        строк {len(preds):>4}  надбавка применена к "
          f"{int(preds['guard_applied'].sum()):>4}  "
          f"сумма HGB {preds['hgb_qty'].sum():.0f} → итог {preds['mu_ml'].sum():.0f}")

In [ ]:
# Пин не совпал — загрузка обязана упасть ДО чтения весов
os.environ["PROGNOZ_STACKED_GUARD_MANIFEST_SHA256"] = "0" * 64
try:
    core.load_bundle(target=TARGET, cfg=core.GuardCfg(lag=3))
    print("!!! бандл с чужим пином загрузился")
except core.BundleError as exc:
    print("бандл с чужим пином отклонён:", exc)
finally:
    os.environ["PROGNOZ_STACKED_GUARD_MANIFEST_SHA256"] = core._manifest_digest(manifest)

## Итог

Полка → панель → признаки → четыре модели (HGB `ml`, CatBoost `catboost`, внутрибандла HGB + q67 + q70 + мета) → OOF → манифест → загрузка по пину → правило.### Отличия от прода| | прод | здесь ||---|---|---|| данные | полка из DuckDB по выгрузке 1С | синтетика || OOF | внешний `PROGNOZ_STACKED_GUARD_OOF` | стадия C, окно 3 мес вместо 9 || версии библиотек | пин `web/requirements.txt` | что стоит в ядре || приёмка бандла | `tools/validate_stacked_bundle.py` | `load_bundle` напрямую || постобработка | `predict/daily.py` | вне ноутбука |### Пределы синтетики- новинки рождаются на всех точках одновременно → сетевая ветка  `novelty_floor` не исполняется, все уходят в холодный старт 8;- `u40_units` и `wo_units` выведены из `supply − demand` (corr 0.99);- цена константна на SKU;- недельный профиль ±6 % против CV спроса ~52 %;- `supply_gap` и `data_missing` независимы по строкам → порог  `NAN_SHARE_FAIL = 0.6` недостижим.### Прогон на реальных данных1. `analysis/etl/10_ingest.py` → `20_marts.py`;2. `tools/build_runtime_shelf.py --as-of=<issue>`;3. стадия C или готовый `oof_with_hgb_provenance.parquet`;4. `PROGNOZ_ML_MODEL_DIR` — песочница;5. `train-stacked-guard.py` + `tools/validate_stacked_bundle.py`, как в   `run-client-nightly.ps1` под `PROGNOZ_NIGHTLY_TRAIN=1`.Артефакты — в `notebooks/_sandbox/`, в git не попадают.